# 2.4 Merge panel data with equipment and FC calculator data

This notebook does the following:
    - Merges the dataset with the equipment calculators

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from fill_missing_mode import fill_with_equipment_mode
from assign_set_temp import assign_set_temp

In [2]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_3.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Load equipment and fc calculations
equipment_calculations = pd.read_csv(
    config.SPARK_DATA / "2_Clean" / "equipment_calculations.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

fc_calculations = pd.read_csv(
    config.SPARK_DATA / "2_Clean" / "fc_calculations.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [4]:
# Load merge overrides file
merge_overrides = pd.read_excel(config.CLEANING_WORKBOOKS / "merge_overrides.xlsx")

## (1) Prepare the equipment_calculations data

In [5]:
# Set all columns to lowercase in equipment_calculations for consistency
equipment_calculations.columns = equipment_calculations.columns.str.lower()

# Rename cols in equipment_calculations to match cleaned equipment dataset for merging
equipment_calculations = equipment_calculations.rename(columns={
    "equipment_type": "equipment",
    "work_surface_width": "width",
    "n_blocks": "blocks",
    "brightness": "monitor_brightness"})

In [6]:
# Check names of equipment in the dataframes to merge
print(equipment["equipment"].unique())
print(equipment_calculations["equipment"].unique())

# Rename values in equipment_calculations to match equipment for merging
equipment_calculations["equipment"] = equipment_calculations["equipment"].replace({
    "heat_block": "heater",
    "it_equipment": "it",
    "microbial_safety_cabinet": "microbio",
    "water_bath": "bath",
    "freezer_20": "freezer",
    "ult_freezer": "ult",
    "drying_cabinet": "glassware",
    "co2_incubator": "incubator"
})

['heater' 'it' 'bath' 'fc' 'freezer' 'fridge' 'microbio' 'ult' 'cryostat'
 'glassware' 'incubator']
['water_bath' 'cryostat' 'it_equipment' 'microbial_safety_cabinet'
 'fridge' 'heat_block' 'drying_cabinet' 'co2_incubator' 'freezer_20'
 'ult_freezer']


In [7]:
# Ensure that merge cols are of same type without turning missings into the string "nan"
merge_cols_string = ["equipment", "type", "size", "age", "monitor_brightness"]
merge_cols_numeric = ["width", "blocks", "set_temp"]

for col in merge_cols_string:
    equipment[col] = equipment[col].astype("string")
    equipment_calculations[col] = equipment_calculations[col].astype("string")

for col in merge_cols_numeric:
    equipment[col] = pd.to_numeric(equipment[col], errors="raise")
    equipment_calculations[col] = pd.to_numeric(equipment_calculations[col], errors="raise")


In [8]:
# Partition equipment_calculations into IT rows and non-IT rows
it_rows = equipment_calculations[equipment_calculations["equipment"] == "it"].copy()
non_it_rows = equipment_calculations[
    (equipment_calculations["equipment"] != "it")
].copy()
non_it_rows.drop(columns=["monitor_brightness"], inplace=True)

## (2) Merge with equipment calculators

### (2a) Merge with equipment calculations (everything apart from IT and FCs)

In [9]:
# Merge equipment with equipment_calculations on "equipment", "type", "width", "size", "blocks", "age", "set_temp"
merge_cols_1 = ["equipment", "type", "width", "size", "blocks", "age", "set_temp"]

# Check for duplicates in non_it_rows for the merge columns before merging, as this would cause a many-to-many merge and duplicate rows in the output.
duplicates = non_it_rows.duplicated(subset=merge_cols_1, keep=False)
if duplicates.any():
    print("Warning: There are duplicates in non_it_rows for the merge columns.")
    print(non_it_rows[duplicates].sort_values(merge_cols_1))


In [10]:
# Merge - standard merge, and separate handling for rows with merge overrides.
strategy_col = next((c for c in ["match_strategy", "merge_strategy"] if c in merge_overrides.columns), None)
if strategy_col is None:
    raise KeyError("merge_overrides must contain 'match_strategy' or 'merge_strategy'.")

override_value_cols = [c for c in merge_overrides.columns if c.startswith("override_")]
overrides = merge_overrides[merge_cols_1 + [strategy_col] + override_value_cols].drop_duplicates()

# Tag each equipment row with its override strategy (NaN = standard merge)
eq = equipment.reset_index(drop=False).rename(columns={"index": "__row_id"})
eq = eq.merge(overrides, on=merge_cols_1, how="left")

standard_rows = eq[eq[strategy_col].isna()]
override_rows = eq[eq[strategy_col].notna()]

pieces = [standard_rows.merge(non_it_rows, on=merge_cols_1, how="left", indicator=True)]

for strategy, group in override_rows.groupby(strategy_col):
    group = group.copy()
    if strategy.startswith("ignore_"):
        ignore_cols = [c for c in strategy.removeprefix("ignore_").split("_") if c in merge_cols_1]
        if not ignore_cols:
            raise ValueError(f"Strategy '{strategy}' does not reference a valid merge column.")
        merged = group.merge(
            non_it_rows,
            on=[c for c in merge_cols_1 if c not in ignore_cols],
            how="left",
            indicator=True,
            suffixes=("_original", "_calculator"),
        )
    else:
        prefix = "use_override_" if strategy.startswith("use_override_") else "override_"
        target_cols = [c for c in strategy.removeprefix(prefix).split("_") if c in merge_cols_1]
        if not target_cols:
            raise ValueError(f"Strategy '{strategy}' does not reference a valid merge column.")
        for col in target_cols:
            group[f"{col}_original"] = group[col]
            group[col] = group[f"override_{col}"]
        merged = group.merge(non_it_rows, on=merge_cols_1, how="left", indicator=True)
    pieces.append(merged)

equipment_1 = pd.concat(pieces, ignore_index=True).drop(
    columns=["__row_id", strategy_col] + override_value_cols, errors="ignore"
)

In [11]:
# Check where merges failed - which equipment types, which combinations of merge columns, and why
key_cols = merge_cols_1
print(equipment_1["_merge"].value_counts())
print()

summary_rows = []
for eq_type in equipment["equipment"].dropna().unique():
    eq_mask = equipment_1["equipment"] == eq_type
    eq_rows = equipment_1[eq_mask]
    left_rows = eq_rows[eq_rows["_merge"] == "left_only"]
    both_rows = eq_rows[eq_rows["_merge"] == "both"]

    summary_rows.append({
        "equipment": eq_type,
        "rows": len(eq_rows),
        "both": len(both_rows),
        "left_only": len(left_rows),
        "distinct_left_keys": len(left_rows[key_cols].drop_duplicates()),
    })

summary = pd.DataFrame(summary_rows).sort_values(["left_only"], ascending=False)
print(summary.to_string(index=False))
print()


_merge
both          1294
left_only      775
right_only       0
Name: count, dtype: int64

equipment  rows  both  left_only  distinct_left_keys
       it   583     0        583                  17
       fc   192     0        192                   2
   heater   137   137          0                   0
     bath   142   142          0                   0
  freezer   295   295          0                   0
   fridge   346   346          0                   0
 microbio   115   115          0                   0
      ult   119   119          0                   0
 cryostat    16    16          0                   0
glassware    39    39          0                   0
incubator    85    85          0                   0



In [12]:
# Check that merge succeeded (no left_only rows for non-IT or FC equipment)
non_it_fc_left_only = equipment_1[
    (equipment_1["_merge"] == "left_only") &
    (~equipment_1["equipment"].isin(["it", "fc"]))
]

# Check that the above is empty
assert non_it_fc_left_only.empty, "There are left_only rows that are not IT or FC"

# Drop the "_merge" column as it's no longer needed
equipment_1 = equipment_1.drop(columns=["_merge"])

### (2b) Merge with equipment calculations for IT equipment

In [13]:
# Split IT rows into non-screen and screen
non_screen_it_rows = it_rows[it_rows["type"] != "Computer Screen"].copy()
screen_it_rows = it_rows[it_rows["type"] == "Computer Screen"].copy()

# Rename and keep only relevant cols for non_screen_it_rows
non_screen_it_rows = non_screen_it_rows.rename(columns={"electricity_use_kwh_per_hour": "electricity_non_screen"})
non_screen_it_rows = non_screen_it_rows[["equipment", "type", "electricity_non_screen"]]

# Rename and keep only relevant cols for screen_it_rows
screen_it_rows = screen_it_rows.rename(columns={"electricity_use_kwh_per_hour": "electricity_screen"})
screen_it_rows = screen_it_rows[["equipment", "type", "size", "monitor_brightness", "electricity_screen"]]

In [14]:
# First merge to get the non-screen electricity values
merge_cols_2 = ["equipment", "type"]

equipment_2 = equipment_1.merge(
    non_screen_it_rows,
    on=merge_cols_2,
    how="left",
    suffixes=("", "_non_screen"),
    indicator=True
)

In [15]:
# Check merge status for IT rows
it_rows_2 = equipment_2[equipment_2["equipment"] == "it"]
print(it_rows_2["_merge"].value_counts())

# Check that the left_only IT rows are the ones with "type" == "Screen only"
assert it_rows_2[
    (it_rows_2["_merge"] == "left_only") &
    (it_rows_2["type"] != "Screen only")
].empty, "There are left_only IT rows that do not have type 'Screen only'"

# Drop the "_merge" column as it's no longer needed
equipment_2 = equipment_2.drop(columns=["_merge"])

_merge
both          579
left_only       4
right_only      0
Name: count, dtype: int64


In [16]:
# Now merge to get the screen electricity values
merge_cols_3 = ["equipment", "size", "monitor_brightness"]
equipment_3 = equipment_2.merge(
    screen_it_rows,
    on=merge_cols_3,
    how="left",
    suffixes=("", "_screen"),
    indicator=True
)

In [17]:
# Check merge status for IT rows
it_rows_3 = equipment_3[equipment_3["equipment"] == "it"]
print(it_rows_3["_merge"].value_counts())

# Check that the left_only IT rows are the ones with "size" = "No monitor"
assert it_rows_3[
    (it_rows_3["_merge"] == "left_only") &
    (it_rows_3["size"] != "No monitor")
].empty, "There are left_only IT rows that do not have size 'No monitor'"

_merge
both          522
left_only      61
right_only      0
Name: count, dtype: int64


In [18]:
# For IT replace electricity_use_kwh_per_hour with electricity_non_screen + electricity_screen (fill missing values with 0)
it_mask = equipment_3["equipment"] == "it"

# Replace electricity_screen and electricity_non_screen NaNs with 0 for IT equipment
equipment_3.loc[it_mask, "electricity_screen"] = equipment_3.loc[it_mask, "electricity_screen"].fillna(0)
equipment_3.loc[it_mask, "electricity_non_screen"] = equipment_3.loc[it_mask, "electricity_non_screen"].fillna(0)

equipment_3.loc[it_mask, "electricity_use_kwh_per_hour"] = (
    equipment_3.loc[it_mask, "electricity_non_screen"] + equipment_3.loc[it_mask, "electricity_screen"]
)

In [19]:
# Check that only missing electricity_use_kwh_per_hour values for fc equipment
fc_mask = equipment_3["equipment"] == "fc"
assert equipment_3.loc[
    ~fc_mask, "electricity_use_kwh_per_hour"
    ].isna().sum() == 0, "There are missing electricity_use_kwh_per_hour values for non-FC equipment"

## Clean up and save processed data

In [20]:
# Save processed data
equipment_3.to_csv(config.PROCESSED_DATA / "panel_processed_4.csv", index =False)